# 線性獨立 (Linear Independence)

本筆記本延續前幾章的秩、RREF 與奇異矩陣，專門討論**線性獨立／線性相關**：

1. **定義** — 什麼是線性獨立與線性相關
2. **判定方法** — 齊次方程組、零空間、RREF、秩
3. **幾何意義** — 是否「撐開」更大的空間
4. **特殊情形** — 零向量、平行向量、向量個數過多
5. **與奇異矩陣的關係** — 方陣的行／列獨立 $\Leftrightarrow$ 可逆
6. **綜合練習**

In [1]:
import sympy as sp
from IPython.display import display, Math

def display_large(expr):
    """Display a sympy expression in LaTeX with large font size."""
    display(Math(r"\large{%s}" % sp.latex(expr)))


def display_huge(expr):
    """Display a sympy expression in LaTeX with huge font size."""
    display(Math(r"\huge{%s}" % sp.latex(expr)))

## 1. 什麼是線性獨立？

給定向量 $\mathbf{v}_1, \mathbf{v}_2, \ldots, \mathbf{v}_k$，考慮線性組合等於零：

$$
c_1\mathbf{v}_1 + c_2\mathbf{v}_2 + \cdots + c_k\mathbf{v}_k = \mathbf{0}
$$

| 名稱 | 英文 | 意義 |
|------|------|------|
| **線性獨立** | Linearly independent | **只有** $c_1=c_2=\cdots=c_k=0$ 才能得到 $\mathbf{0}$（平凡解） |
| **線性相關** | Linearly dependent | 存在**不全為零**的係數，使組合仍為 $\mathbf{0}$（非平凡解） |

直觀說法：

- **獨立** = 每個向量都帶來「新方向」，沒有一個是多餘的
- **相關** = 至少有一個向量可以寫成其他向量的線性組合

## 2. 用矩陣統一判定

把向量排成矩陣 $A$ 的**行（columns）**：

$$
A = \begin{bmatrix} \mathbf{v}_1 & \mathbf{v}_2 & \cdots & \mathbf{v}_k \end{bmatrix}
$$

則

$$
c_1\mathbf{v}_1 + \cdots + c_k\mathbf{v}_k = \mathbf{0}
\quad\Leftrightarrow\quad
A\mathbf{c} = \mathbf{0}
$$

因此：

| 條件 | 結論 |
|------|------|
| $A\mathbf{c}=\mathbf{0}$ 只有 $\mathbf{c}=\mathbf{0}$ | 行向量**線性獨立** |
| 零空間有非零向量 | 行向量**線性相關** |
| $\operatorname{rank}(A) = k$（滿行秩） | **獨立** |
| $\operatorname{rank}(A) < k$ | **相關** |

SymPy 常用工具：`A.nullspace()`、`A.rank()`、`A.rref()`。

In [2]:
def check_independence(vectors, names=None):
    """Check linear independence of a list of column vectors."""
    A = sp.Matrix.hstack(*vectors)
    n = A.cols
    r = A.rank()
    rref_A, pivots = A.rref()
    null = A.nullspace()

    if names is None:
        names = [f"v{i+1}" for i in range(n)]

    print("Matrix A with columns", ", ".join(names) + ":")
    display_huge(A)
    print("RREF:")
    display_huge(rref_A)
    print(f"rank(A) = {r}, number of vectors n = {n}")
    print("Pivots:", list(pivots))

    if r == n:
        print("=> Linearly INDEPENDENT (full column rank)")
    else:
        print("=> Linearly DEPENDENT (rank < n)")

    print("Nullspace basis:")
    if not null:
        print("  {0}  (only the trivial solution)")
    else:
        for i, nvec in enumerate(null):
            print(f"  n{i+1} =")
            display_large(nvec)
    print()
    return A

## 3. 例子：獨立 vs 相關（$\mathbb{R}^2$）

### 3.1 線性獨立

$$
\mathbf{v}_1 = \begin{pmatrix} 1 \\ 2 \end{pmatrix},\quad
\mathbf{v}_2 = \begin{pmatrix} 3 \\ 5 \end{pmatrix}
$$

兩者不平行，應為獨立。

In [3]:
v1 = sp.Matrix([1, 2])
v2 = sp.Matrix([3, 5])

A_ind = check_independence([v1, v2], names=["v1", "v2"])

# For 2x2, det != 0 also confirms independence
print("det(A) =")
display_large(A_ind.det())

Matrix A with columns v1, v2:


<IPython.core.display.Math object>

RREF:


<IPython.core.display.Math object>

rank(A) = 2, number of vectors n = 2
Pivots: [0, 1]
=> Linearly INDEPENDENT (full column rank)
Nullspace basis:
  {0}  (only the trivial solution)

det(A) =


<IPython.core.display.Math object>

### 3.2 線性相關

$$
\mathbf{u}_1 = \begin{pmatrix} 1 \\ 2 \end{pmatrix},\quad
\mathbf{u}_2 = \begin{pmatrix} 2 \\ 4 \end{pmatrix} = 2\mathbf{u}_1
$$

第二個是第一個的倍數 → 相關，且 $2\mathbf{u}_1 - 1\cdot\mathbf{u}_2 = \mathbf{0}$。

In [4]:
u1 = sp.Matrix([1, 2])
u2 = sp.Matrix([2, 4])

A_dep = check_independence([u1, u2], names=["u1", "u2"])

print("det(A) =")
display_large(A_dep.det())

# Explicit dependence relation from nullspace
c = A_dep.nullspace()[0]
print("Dependence: c1*u1 + c2*u2 = 0 with c =")
display_large(c)
print("Check:")
display_large(c[0]*u1 + c[1]*u2)

Matrix A with columns u1, u2:


<IPython.core.display.Math object>

RREF:


<IPython.core.display.Math object>

rank(A) = 1, number of vectors n = 2
Pivots: [0]
=> Linearly DEPENDENT (rank < n)
Nullspace basis:
  n1 =


...

<IPython.core.display.Math object>

Dependence: c1*u1 + c2*u2 = 0 with c =


<IPython.core.display.Math object>

Check:


<IPython.core.display.Math object>

## 4. 幾何意義

| 情境 | 幾何圖像 |
|------|----------|
| 兩個獨立向量（$\mathbb{R}^2$） | 不共線，可張成整個平面 |
| 兩個相關向量 | 共線（同一條直線上） |
| 三個獨立向量（$\mathbb{R}^3$） | 不共面，可張成整個空間 |
| 三個相關向量 | 落在同一個平面（或更低維）上 |

對方陣而言，$\lvert\det(A)\rvert$ 是以各行為邊的平行四邊形／平行多面體體積：

- $\det \neq 0$ → 有正體積 → 行向量獨立
- $\det = 0$ → 體積塌縮 → 行向量相關

In [5]:
print("Independent pair: area |det| =")
display_large(sp.Abs(A_ind.det()))

print("Dependent pair: area |det| =")
display_large(sp.Abs(A_dep.det()))

Independent pair: area |det| =


<IPython.core.display.Math object>

Dependent pair: area |det| =


<IPython.core.display.Math object>

## 5. $\mathbb{R}^3$ 的例子

### 5.1 三個獨立向量

$$
\mathbf{a}_1 = \begin{pmatrix} 1 \\ 0 \\ 0 \end{pmatrix},\quad
\mathbf{a}_2 = \begin{pmatrix} 1 \\ 1 \\ 0 \end{pmatrix},\quad
\mathbf{a}_3 = \begin{pmatrix} 1 \\ 1 \\ 1 \end{pmatrix}
$$

In [6]:
a1 = sp.Matrix([1, 0, 0])
a2 = sp.Matrix([1, 1, 0])
a3 = sp.Matrix([1, 1, 1])

A3 = check_independence([a1, a2, a3], names=["a1", "a2", "a3"])
print("det(A) =")
display_large(A3.det())

Matrix A with columns a1, a2, a3:


<IPython.core.display.Math object>

RREF:


<IPython.core.display.Math object>

rank(A) = 3, number of vectors n = 3
Pivots: [0, 1, 2]
=> Linearly INDEPENDENT (full column rank)
Nullspace basis:
  {0}  (only the trivial solution)

det(A) =


<IPython.core.display.Math object>

### 5.2 三個相關向量（其中一個是另外兩個的組合）

$$
\mathbf{b}_1 = \begin{pmatrix} 1 \\ 2 \\ 3 \end{pmatrix},\quad
\mathbf{b}_2 = \begin{pmatrix} 4 \\ 5 \\ 6 \end{pmatrix},\quad
\mathbf{b}_3 = \begin{pmatrix} 7 \\ 8 \\ 9 \end{pmatrix}
$$

注意 $\mathbf{b}_3 = 2\mathbf{b}_2 - \mathbf{b}_1$（與奇異矩陣章節同一例子）。

In [7]:
b1 = sp.Matrix([1, 2, 3])
b2 = sp.Matrix([4, 5, 6])
b3 = sp.Matrix([7, 8, 9])

B = check_independence([b1, b2, b3], names=["b1", "b2", "b3"])

print("Verify b3 == 2*b2 - b1:")
display_large(2*b2 - b1)

c = B.nullspace()[0]
print("Dependence coefficients c =")
display_large(c)
print("c1*b1 + c2*b2 + c3*b3 =")
display_large(c[0]*b1 + c[1]*b2 + c[2]*b3)

Matrix A with columns b1, b2, b3:


<IPython.core.display.Math object>

RREF:


<IPython.core.display.Math object>

rank(A) = 2, number of vectors n = 3
Pivots: [0, 1]
=> Linearly DEPENDENT (rank < n)
Nullspace basis:
  n1 =


...

<IPython.core.display.Math object>

Dependence coefficients c =


<IPython.core.display.Math object>

c1*b1 + c2*b2 + c3*b3 =


<IPython.core.display.Math object>

## 6. 特殊情形

### 6.1 含零向量 → 一定相關

因為 $1\cdot\mathbf{0} + 0\cdot\mathbf{v}_2 + \cdots = \mathbf{0}$，係數不全為零。

In [8]:
z = sp.Matrix([0, 0])
w = sp.Matrix([1, 3])
check_independence([z, w], names=["0", "w"]);

Matrix A with columns 0, w:


<IPython.core.display.Math object>

RREF:


<IPython.core.display.Math object>

rank(A) = 1, number of vectors n = 2
Pivots: [1]
=> Linearly DEPENDENT (rank < n)
Nullspace basis:
  n1 =


<IPython.core.display.Math object>

### 6.2 單一非零向量 → 獨立；單一零向量 → 相關

對單一向量 $\{\mathbf{v}\}$：$c\mathbf{v}=\mathbf{0}$ 只有在 $\mathbf{v}\neq\mathbf{0}$ 時才迫使 $c=0$。

In [9]:
print("--- nonzero singleton ---")
check_independence([sp.Matrix([2, -1])], names=["v"])

print("--- zero singleton ---")
check_independence([sp.Matrix([0, 0])], names=["0"]);

--- nonzero singleton ---
Matrix A with columns v:


<IPython.core.display.Math object>

RREF:


<IPython.core.display.Math object>

rank(A) = 1, number of vectors n = 1
Pivots: [0]
=> Linearly INDEPENDENT (full column rank)
Nullspace basis:
  {0}  (only the trivial solution)

--- zero singleton ---
Matrix A with columns 0:


<IPython.core.display.Math object>

RREF:


<IPython.core.display.Math object>

rank(A) = 0, number of vectors n = 1
Pivots: []
=> Linearly DEPENDENT (rank < n)
Nullspace basis:
  n1 =


<IPython.core.display.Math object>

### 6.3 向量個數超過空間維度 → 一定相關

在 $\mathbb{R}^n$ 中，任取 $k > n$ 個向量，必線性相關（矩陣最多只有 $n$ 個主元，無法滿行秩）。

例如在 $\mathbb{R}^2$ 放三個向量：

In [10]:
p1 = sp.Matrix([1, 0])
p2 = sp.Matrix([0, 1])
p3 = sp.Matrix([1, 1])

P = check_independence([p1, p2, p3], names=["p1", "p2", "p3"])

# Express dependence: p3 = p1 + p2  =>  p1 + p2 - p3 = 0
c = P.nullspace()[0]
print("One dependence relation uses c =")
display_large(c)

Matrix A with columns p1, p2, p3:


<IPython.core.display.Math object>

RREF:


<IPython.core.display.Math object>

rank(A) = 2, number of vectors n = 3
Pivots: [0, 1]
=> Linearly DEPENDENT (rank < n)
Nullspace basis:
  n1 =


<IPython.core.display.Math object>


One dependence relation uses c =


<IPython.core.display.Math object>

## 7. 與奇異矩陣、可逆性的關係

對 **$n \times n$ 方陣** $A$，以下全部等價：

1. $A$ 的行向量線性獨立
2. $A$ 的列向量線性獨立
3. $\operatorname{rank}(A) = n$
4. $\det(A) \neq 0$
5. $A$ **非奇異**（可逆）
6. $A\mathbf{x}=\mathbf{0}$ 只有平凡解

這把本章與前一章「奇異矩陣」連起來：奇異 $\Leftrightarrow$ 行／列線性相關。

In [11]:
def square_independence_report(A, name="A"):
    """Link independence checks to singularity for a square matrix."""
    print(f"=== {name} ===")
    display_huge(A)
    det_A = A.det()
    print("det =")
    display_large(det_A)
    print(f"rank = {A.rank()} (n = {A.rows})")
    print("Columns independent?", A.rank() == A.cols)
    print("Singular?", det_A == 0)
    print("Invertible?", det_A != 0)
    print()


square_independence_report(A_ind, "independent 2x2")
square_independence_report(A_dep, "dependent 2x2")
square_independence_report(B, "dependent 3x3")

=== independent 2x2 ===


<IPython.core.display.Math object>

det =


<IPython.core.display.Math object>

rank = 2 (n = 2)
Columns independent? True
Singular? False
Invertible? True

=== dependent 2x2 ===


...

rank = 1 (n = 2)
Columns independent? False
Singular? True
Invertible? False

=== dependent 3x3 ===


<IPython.core.display.Math object>

det =


<IPython.core.display.Math object>

rank = 2 (n = 3)
Columns independent? False
Singular? True
Invertible? False



## 8. 用 RREF 找出「多餘」的向量

RREF 的主元行對應**獨立子集**；沒有主元的行是多餘的，可寫成主元行的組合。

下面四個 $\mathbb{R}^3$ 向量中，哪些獨立？

In [12]:
v1 = sp.Matrix([1, 2, 0])
v2 = sp.Matrix([2, 4, 0])   # = 2*v1
v3 = sp.Matrix([0, 1, 1])
v4 = sp.Matrix([1, 3, 1])   # = v1 + v3

M = sp.Matrix.hstack(v1, v2, v3, v4)
print("M =")
display_huge(M)

rref_M, pivots = M.rref()
print("RREF =")
display_huge(rref_M)
print("Pivot columns (0-based):", list(pivots))
print(f"rank = {M.rank()} => a maximal independent subset has {M.rank()} vectors")
print("Independent subset: columns", [i + 1 for i in pivots], "(1-based)")

# Show free-column dependence via nullspace
print("\nNullspace (each basis vector is one dependence relation):")
for i, nvec in enumerate(M.nullspace()):
    print(f"n{i+1} =")
    display_large(nvec)

M =


<IPython.core.display.Math object>

RREF =


<IPython.core.display.Math object>

Pivot columns (0-based): [0, 2]
rank = 2 => a maximal independent subset has 2 vectors
Independent subset: columns [1, 3] (1-based)

Nullspace (each basis vector is one dependence relation):
n1 =


<IPython.core.display.Math object>

n2 =


<IPython.core.display.Math object>

## 9. 綜合練習

考慮

$$
\mathbf{w}_1 = \begin{pmatrix} 1 \\ 1 \\ 2 \end{pmatrix},\quad
\mathbf{w}_2 = \begin{pmatrix} 2 \\ 3 \\ 1 \end{pmatrix},\quad
\mathbf{w}_3 = \begin{pmatrix} 4 \\ 5 \\ 5 \end{pmatrix}
$$

請完成：

1. 判斷 $\{\mathbf{w}_1,\mathbf{w}_2,\mathbf{w}_3\}$ 是否線性獨立
2. 若相關，寫出一組非平凡係數使 $c_1\mathbf{w}_1+c_2\mathbf{w}_2+c_3\mathbf{w}_3=\mathbf{0}$
3. 計算 $\operatorname{rank}$，並指出一組最大獨立子集
4. 說明對應方陣是否奇異

In [13]:
w1 = sp.Matrix([1, 1, 2])
w2 = sp.Matrix([2, 3, 1])
w3 = sp.Matrix([4, 5, 5])

W = check_independence([w1, w2, w3], names=["w1", "w2", "w3"])

rref_W, pivots_W = W.rref()
print("Maximal independent subset: columns", [i + 1 for i in pivots_W], "(1-based)")

if W.nullspace():
    c = W.nullspace()[0]
    print("Nontrivial coefficients c =")
    display_large(c)
    print("Verification:")
    display_large(c[0]*w1 + c[1]*w2 + c[2]*w3)

print("Singular (as square matrix)?", W.det() == 0)
print("det(W) =")
display_large(W.det())

Matrix A with columns w1, w2, w3:


<IPython.core.display.Math object>

RREF:


<IPython.core.display.Math object>

rank(A) = 2, number of vectors n = 3
Pivots: [0, 1]
=> Linearly DEPENDENT (rank < n)
Nullspace basis:
  n1 =


...

<IPython.core.display.Math object>

Verification:


<IPython.core.display.Math object>

Singular (as square matrix)? True
det(W) =


<IPython.core.display.Math object>

## 小結

| 概念 | SymPy | 重點 |
|------|-------|------|
| 組成矩陣 | `Matrix.hstack(v1, v2, ...)` | 向量當行 |
| 獨立判定 | `A.rank() == A.cols` | 滿行秩 $\Leftrightarrow$ 獨立 |
| 相關關係 | `A.nullspace()` | 非零核向量 = 非平凡係數 |
| RREF / 主元 | `A.rref()` | 主元行 = 最大獨立子集 |
| 方陣特例 | `A.det() != 0` | 獨立 $\Leftrightarrow$ 非奇異 |

**記住：**

- 線性獨立 $\Leftrightarrow$ $A\mathbf{c}=\mathbf{0}$ 只有 $\mathbf{c}=\mathbf{0}$
- 線性相關 $\Leftrightarrow$ 至少有一個向量是其他向量的組合
- 含 $\mathbf{0}$、或 $k > n$（在 $\mathbb{R}^n$）→ 一定相關
- 對 $n\times n$ 方陣：行獨立 = 列獨立 = 可逆 = $\det\neq 0$